In [0]:
import time
import pandas as pd
from datetime import datetime
from snowflake.snowpark.context import get_active_session

session = get_active_session()

CSV_FILE_PATH = "snow://workspace/JOEL.PUBLIC.\"benchmark\"/versions/head/benchmark_queries/final.csv"
DEFAULT_DATABASE = "apjtechup"
DEFAULT_SCHEMA = "gold_dimensional"
NUM_ITERATIONS = 3

print(f"Reading queries from: {CSV_FILE_PATH}")
print(f"Default database: {DEFAULT_DATABASE}")
print(f"Default schema: {DEFAULT_SCHEMA}")
print(f"Number of iterations: {NUM_ITERATIONS}")
print("="*80)

In [0]:
import io
import time

file_stream = session.file.get_stream(CSV_FILE_PATH)
content = file_stream.read().decode('utf-8')
queries = [line.strip() for line in content.split('\n') if line.strip()]

print(f"\nLoaded {len(queries)} queries from file\n")

In [0]:
session.sql(f"alter session set USE_CACHED_RESULT=FALSE").collect()


session.sql(f"USE DATABASE {DEFAULT_DATABASE}").collect()
session.sql(f"USE SCHEMA {DEFAULT_SCHEMA}").collect()

async_jobs = []
submission_results = []

for iteration in range(1, NUM_ITERATIONS + 1):
    print(f"\n{'='*80}")
    print(f"ITERATION {iteration}/{NUM_ITERATIONS}")
    print(f"{'='*80}")
    
    for idx, query in enumerate(queries, 1):
        print(f"Submitting query {idx}/{len(queries)} (iteration {iteration})...")
        
        try:
            submit_time = datetime.now()
            async_job = session.sql(query).collect_nowait()
            
            submission_results.append({
                "iteration": iteration,
                "query_index": idx,
                "query": query[:100] + "..." if len(query) > 100 else query,
                "query_id": async_job.query_id,
                "submit_time": submit_time,
                "status": "SUBMITTED"
            })
            async_jobs.append(async_job)
            print(f"  Query {idx} submitted. Query ID: {async_job.query_id}")
            
        except Exception as e:
            print(f"  Error submitting query {idx}: {str(e)}")
            submission_results.append({
                "iteration": iteration,
                "query_index": idx,
                "query": query[:100] + "..." if len(query) > 100 else query,
                "query_id": None,
                "submit_time": datetime.now(),
                "status": f"SUBMIT_ERROR: {str(e)}"
            })
    
    print(f"\nIteration {iteration} complete. Submitted {len(queries)} queries.")
    time.sleep(10)
    print("Waiting 10 seconds before next batch")


print(f"\nTotal submitted: {len(async_jobs)} queries across {NUM_ITERATIONS} iterations")

In [0]:
query_ids = [job.query_id for job in async_jobs]
query_ids_str = "','".join(query_ids)

print("\nWaiting for all queries to complete...")
print("="*80)

while True:
    running_check = session.sql(f"""
        SELECT COUNT(*) as running_count
        FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY(RESULT_LIMIT => 1000))
        WHERE QUERY_ID IN ('{query_ids_str}')
          AND EXECUTION_STATUS = 'RUNNING'
    """).collect()
    
    running_count = running_check[0]['RUNNING_COUNT']
    print(f"  Queries still running: {running_count}")
    
    if running_count == 0:
        break
    time.sleep(5)

print("\nAll queries completed. Fetching results...")

query_history = session.sql(f"""
    SELECT 
        QUERY_ID,
        START_TIME,
        END_TIME,
        TOTAL_ELAPSED_TIME,
        EXECUTION_STATUS,
        ROWS_PRODUCED,
        ERROR_MESSAGE
    FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY(RESULT_LIMIT => 1000))
    WHERE QUERY_ID IN ('{query_ids_str}')
""").collect()

history_map = {row['QUERY_ID']: row.as_dict() for row in query_history}

final_results = []
for submission in submission_results:
    if submission["status"] != "SUBMITTED":
        final_results.append(submission)
        continue
    
    query_id = submission["query_id"]
    hist = history_map.get(query_id, {})
    
    if hist:
        exec_status = hist.get('EXECUTION_STATUS', 'UNKNOWN')
        status = "SUCCESS" if exec_status == 'SUCCESS' else f"EXECUTION_ERROR: {hist.get('ERROR_MESSAGE', exec_status)}"
        execution_time_ms = hist.get('TOTAL_ELAPSED_TIME')
        execution_time_seconds = execution_time_ms / 1000.0 if execution_time_ms else None
        
        final_results.append({
            **submission,
            "start_time": hist.get('START_TIME'),
            "end_time": hist.get('END_TIME'),
            "execution_time_ms": execution_time_ms,
            "execution_time_seconds": execution_time_seconds,
            "row_count": hist.get('ROWS_PRODUCED'),
            "status": status
        })
    else:
        final_results.append({
            **submission,
            "status": "HISTORY_NOT_FOUND"
        })

print(f"Processed {len(final_results)} query results.")
print("\n" + "="*80)
print("All queries completed!")

In [0]:
results_df = pd.DataFrame(final_results)

session.sql("USE DATABASE JOEL").collect()
session.sql("USE SCHEMA PUBLIC").collect()

snowpark_df = session.create_dataframe(results_df)
snowpark_df.create_or_replace_temp_view("BENCHMARK_RESULTS")

# Rename columns to match DBX format
results_df = results_df.rename(columns={
    'iteration': 'loop_iteration',
    'query_index': 'query_number',
    'query_id': 'statement_id',
    'execution_time_ms': 'duration_ms',
    'execution_time_seconds': 'duration_seconds',
    'row_count': 'rows_produced'
})

# Add error_message column for consistency
results_df['error_message'] = results_df['status'].apply(
    lambda x: x.replace('EXECUTION_ERROR: ', '') if x.startswith('EXECUTION_ERROR:') else None
)
results_df['status'] = results_df['status'].apply(
    lambda x: 'SUCCEEDED' if x == 'SUCCESS' else ('FAILED' if x.startswith('EXECUTION_ERROR') else x)
)

print(f"{'='*80}")
print(f"BENCHMARK SUMMARY")
print(f"{'='*80}\n")

success_count = len(results_df[results_df['status'] == 'SUCCEEDED'])
error_count = len(results_df[results_df['status'] != 'SUCCEEDED'])

print(f"Total executions: {len(results_df)}")
print(f"Total loops: {NUM_ITERATIONS}")
print(f"Queries per loop: {len(queries)}")
print(f"Succeeded: {success_count}")
print(f"Failed: {error_count}")

# Calculate min start time and max end time across all loops
if 'start_time' in results_df.columns and results_df['start_time'].notna().any():
    start_times = pd.to_datetime(results_df['start_time'].dropna(), errors='coerce')
    end_times = pd.to_datetime(results_df['end_time'].dropna(), errors='coerce')
    
    min_start_time = start_times.min()
    max_end_time = end_times.max()
    total_elapsed_time = (max_end_time - min_start_time).total_seconds()
    
    print(f"\nOverall Timing Summary (all loops):")
    print(f"  Minimum start time: {min_start_time}")
    print(f"  Maximum end time: {max_end_time}")
    print(f"  Total elapsed time (start to end): {total_elapsed_time:.2f} seconds ({total_elapsed_time/60:.2f} minutes)")

if 'duration_seconds' in results_df.columns and results_df['duration_seconds'].notna().any():
    print(f"\nQuery Duration Statistics (all loops):")
    print(f"  Average duration: {results_df['duration_seconds'].mean():.2f} seconds")
    print(f"  Min duration: {results_df['duration_seconds'].min():.2f} seconds")
    print(f"  Max duration: {results_df['duration_seconds'].max():.2f} seconds")
    print(f"  Total duration (sum): {results_df['duration_seconds'].sum():.2f} seconds")

if 'rows_produced' in results_df.columns and results_df['rows_produced'].notna().any():
    print(f"\nRows Produced Statistics (all loops):")
    print(f"  Total rows produced: {results_df['rows_produced'].sum():.0f}")
    print(f"  Average rows per query: {results_df['rows_produced'].mean():.0f}")

# Per-loop statistics
print(f"\n{'='*80}")
print("Per-Loop Statistics:")
print(f"{'='*80}")
for loop_num in range(1, NUM_ITERATIONS + 1):
    loop_df = results_df[results_df['loop_iteration'] == loop_num]
    if len(loop_df) > 0 and loop_df['duration_seconds'].notna().any():
        print(f"\nLoop {loop_num}:")
        print(f"  Queries executed: {len(loop_df)}")
        print(f"  Succeeded: {len(loop_df[loop_df['status'] == 'SUCCEEDED'])}")
        print(f"  Failed: {len(loop_df[loop_df['status'].isin(['FAILED', 'SUBMISSION_FAILED'])])}")
        print(f"  Average duration: {loop_df['duration_seconds'].mean():.2f} seconds")
        print(f"  Total duration: {loop_df['duration_seconds'].sum():.2f} seconds")
        
        if loop_df['start_time'].notna().any() and loop_df['end_time'].notna().any():
            loop_start = pd.to_datetime(loop_df['start_time'].dropna(), errors='coerce').min()
            loop_end = pd.to_datetime(loop_df['end_time'].dropna(), errors='coerce').max()
            loop_elapsed = (loop_end - loop_start).total_seconds()
            print(f"  Loop elapsed time: {loop_elapsed:.2f} seconds ({loop_elapsed/60:.2f} minutes)")

print(f"\n{'='*80}")

In [0]:
import matplotlib.pyplot as plt
import numpy as np

loop_count = results_df['loop_iteration'].max()

loop_metrics = []
for loop_num in range(1, loop_count + 1):
    loop_df = results_df[results_df['loop_iteration'] == loop_num]
    
    if len(loop_df) > 0:
        total_runtime = loop_df['duration_seconds'].sum() if loop_df['duration_seconds'].notna().any() else 0
        
        wall_time = 0
        if loop_df['start_time'].notna().any() and loop_df['end_time'].notna().any():
            loop_start = pd.to_datetime(loop_df['start_time'].dropna(), errors='coerce').min()
            loop_end = pd.to_datetime(loop_df['end_time'].dropna(), errors='coerce').max()
            wall_time = (loop_end - loop_start).total_seconds()
        
        loop_metrics.append({
            'loop': loop_num,
            'total_runtime_seconds': total_runtime,
            'wall_time_seconds': wall_time,
            'total_runtime_minutes': total_runtime / 60,
            'wall_time_minutes': wall_time / 60
        })

metrics_df = pd.DataFrame(loop_metrics)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.bar(metrics_df['loop'], metrics_df['total_runtime_minutes'], color='steelblue', alpha=0.8)
ax1.set_xlabel('Loop Iteration', fontsize=12, fontweight='bold')
ax1.set_ylabel('Total Runtime (minutes)', fontsize=12, fontweight='bold')
ax1.set_title('Total Query Runtime per Loop\n(Sum of All Query Durations)', fontsize=13, fontweight='bold')
ax1.grid(axis='y', alpha=0.3, linestyle='--')
ax1.set_xticks(metrics_df['loop'])

for i, (loop, runtime) in enumerate(zip(metrics_df['loop'], metrics_df['total_runtime_minutes'])):
    ax1.text(loop, runtime + (ax1.get_ylim()[1] * 0.02), f'{runtime:.2f}m', 
             ha='center', va='bottom', fontsize=10, fontweight='bold')

ax2.bar(metrics_df['loop'], metrics_df['wall_time_minutes'], color='coral', alpha=0.8)
ax2.set_xlabel('Loop Iteration', fontsize=12, fontweight='bold')
ax2.set_ylabel('Wall Time (minutes)', fontsize=12, fontweight='bold')
ax2.set_title('End-to-End Wall Time per Loop\n(Min Start → Max End Time)', fontsize=13, fontweight='bold')
ax2.grid(axis='y', alpha=0.3, linestyle='--')
ax2.set_xticks(metrics_df['loop'])

for i, (loop, wall_time) in enumerate(zip(metrics_df['loop'], metrics_df['wall_time_minutes'])):
    ax2.text(loop, wall_time + (ax2.get_ylim()[1] * 0.02), f'{wall_time:.2f}m', 
             ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.suptitle(f'Benchmark Performance Metrics ({loop_count} Loop{"s" if loop_count > 1 else ""})', 
             fontsize=15, fontweight='bold', y=1.02)
plt.show()

print("\n" + "="*80)
print("LOOP TIMING SUMMARY")
print("="*80)
print(f"{'Loop':<8} {'Total Runtime':<20} {'Wall Time':<20} {'Parallelism Factor':<20}")
print("-"*80)

for _, row in metrics_df.iterrows():
    loop = int(row['loop'])
    runtime_min = row['total_runtime_minutes']
    wall_min = row['wall_time_minutes']
    parallelism = runtime_min / wall_min if wall_min > 0 else 0
    
    print(f"{loop:<8} {runtime_min:>8.2f} min ({row['total_runtime_seconds']:>7.1f}s)  "
          f"{wall_min:>8.2f} min ({row['wall_time_seconds']:>7.1f}s)  "
          f"{parallelism:>8.2f}x")

print("="*80)
print("\nNote: Parallelism Factor = Total Runtime / Wall Time")
print("      Higher values indicate more parallel query execution")

In [0]:
print("\n" + "="*80)
print("DETAILED RESULTS")
print("="*80 + "\n")

display_columns = ['loop_iteration', 'query_number', 'statement_id', 'status', 
                   'duration_seconds', 'rows_produced']

available_cols = [col for col in display_columns if col in results_df.columns]
results_display = results_df[available_cols].copy()

results_display

In [0]:
print(f"\n{'='*80}")
print(f"BENCHMARK COMPLETE")
print(f"{'='*80}")